Import the data

In [1]:
import pandas as pd

df = pd.read_csv("data/restructured.csv")

In [2]:
df.head()

,roundup_title,topics,story_title,story_text,bias_label
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center
1,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left
2,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right
4,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center


Preprocess the Data

In [3]:
df['full_text'] = df['story_title'] + " " + df['story_text']

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download stopwords from NLTK
#nltk.download('punkt')
#nltk.download('stopwords')

def data_preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    
    new_tokens = []
    for w in tokens:
        if w not in stop_words and w.isalpha():  
            stem_tokens = ps.stem(w)
            new_tokens.append(stem_tokens)
    return new_tokens


# Apply preprocessing to each message
df['full_text'] = df['full_text'].apply(data_preprocess)

# Preview the data to ensure pre-processing is as expected
df.head(10)


,roundup_title,topics,story_title,story_text,bias_label,full_text
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center,"[thousand, gather, mark, civil, right, march, ..."
1,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left,"[march, washington, continu, focu, civil, righ..."
2,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right,"[rememb, uncl, fifti, year, ago, valiant, grou..."
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right,"[presid, trump, call, good, shutdown, septemb,..."
4,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center,"[trump, us, need, good, shutdown, presid, trum..."
5,"""Good Shutdown"" in September?",Politics,Trump's frustration with budget compromise has...,Congress looks set to enact a bipartisan spend...,left,"[trump, frustrat, budget, compromis, consid, m..."
6,"""Skinny Repeal"" Motions Fails",US Senate,Obamacare repeal is dead for now. What could t...,The Senate’s effort to repeal and replace Obam...,center,"[obamacar, repeal, dead, could, mean, senat, e..."
7,"""Skinny Repeal"" Motions Fails",US Senate,Why Senate Republicans couldn’t repeal Obamacare,"In a stunning turn, Senate Republicans — in th...",left,"[senat, republican, repeal, obamacar, stun, tu..."
8,"""Skinny Repeal"" Motions Fails",US Senate,Senate Fails To Pass Motion To Proceed On ‘Ski...,Senate Majority Leader Mitch McConnell failed ...,right,"[senat, fail, pass, motion, proceed, skinni, r..."
9,"""Trump University"" Documents Unsealed",Elections,Trump University ‘Playbooks’ Unsealed in Lawsu...,Trump University gave employees detailed instr...,right,"[trump, univers, playbook, unseal, lawsuit, se..."


Create Dataframes for binary and multi-class classification

In [5]:
df_multi = df.copy()

# DF for binary classification (only left and right)
df_binary = df[df['bias_label'].isin(['left', 'right'])].copy()
df_binary.reset_index(drop=True, inplace=True)


Split the data into train and test sets (to avoid data leakage when applying bag of words and TF-IDF)

In [6]:
ml_list_multi = list(zip(df_multi['full_text'], df_multi['bias_label']))
ml_list_binary = list(zip(df_binary['full_text'], df_binary['bias_label']))

size_multi = int(len(ml_list_multi) * 0.75)
train_multi, test_multi = ml_list_multi[:size_multi], ml_list_multi[size_multi:]

size_binary = int(len(ml_list_binary) * 0.75)
train_binary, test_binary = ml_list_binary[:size_binary], ml_list_binary[size_binary:]

Shared Setup

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd

# Convert tokenized docs to text
df_multi['text_str'] = df_multi['full_text'].apply(lambda tokens: ' '.join(tokens))
df_binary['text_str'] = df_binary['full_text'].apply(lambda tokens: ' '.join(tokens))

# Recreate X/y splits
X_train_multi = [' '.join(tokens) for tokens, label in train_multi]
y_train_multi = [label for tokens, label in train_multi]
X_test_multi = [' '.join(tokens) for tokens, label in test_multi]
y_test_multi = [label for tokens, label in test_multi]

X_train_binary = [' '.join(tokens) for tokens, label in train_binary]
y_train_binary = [label for tokens, label in train_binary]
X_test_binary = [' '.join(tokens) for tokens, label in test_binary]
y_test_binary = [label for tokens, label in test_binary]

1a Bernoulli Naive Bayes + Binary BoW (Multi-class)

In [8]:
vectorizer = CountVectorizer(binary=True, max_features=1000, ngram_range=(1, 1))
X_train_vec = vectorizer.fit_transform(X_train_multi)
X_test_vec = vectorizer.transform(X_test_multi)

model = BernoulliNB()
model.fit(X_train_vec, y_train_multi)
y_pred = model.predict(X_test_vec)

print("\n--- BernoulliNB + Binary BoW (Multi-class) ---")
print(f"Accuracy: {accuracy_score(y_test_multi, y_pred):.2%}")
print(classification_report(y_test_multi, y_pred))
print(pd.DataFrame(confusion_matrix(y_test_multi, y_pred),
                   index=model.classes_, columns=model.classes_))


--- BernoulliNB + Binary BoW (Multi-class) ---
Accuracy: 44.12%
              precision    recall  f1-score   support

      center       0.42      0.37      0.39      1941
        left       0.44      0.48      0.46      2098
       right       0.46      0.47      0.47      2088

    accuracy                           0.44      6127
   macro avg       0.44      0.44      0.44      6127
weighted avg       0.44      0.44      0.44      6127

        center  left  right
center     713   657    571
left       499  1001    598
right      493   606    989


1b Bernoulli Naive Bayes + Binary BoW (Binary)

In [9]:
vectorizer = CountVectorizer(binary=True, max_features=1000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train_binary)
X_test_vec = vectorizer.transform(X_test_binary)

model = BernoulliNB()
model.fit(X_train_vec, y_train_binary)
y_pred = model.predict(X_test_vec)

print("\n--- BernoulliNB + Binary BoW (Binary) ---")
print(f"Accuracy: {accuracy_score(y_test_binary, y_pred):.2%}")
print(classification_report(y_test_binary, y_pred))
print(pd.DataFrame(confusion_matrix(y_test_binary, y_pred),
                   index=model.classes_, columns=model.classes_))


--- BernoulliNB + Binary BoW (Binary) ---
Accuracy: 61.57%
              precision    recall  f1-score   support

        left       0.61      0.63      0.62      2106
       right       0.62      0.60      0.61      2096

    accuracy                           0.62      4202
   macro avg       0.62      0.62      0.62      4202
weighted avg       0.62      0.62      0.62      4202

       left  right
left   1330    776
right   839   1257


2a Multinomial Naive Bayes + Count BoW (Multi-class)

In [10]:
vectorizer = CountVectorizer(max_features=1000, ngram_range=(1, 2))  # default binary=False
X_train_vec = vectorizer.fit_transform(X_train_multi)
X_test_vec = vectorizer.transform(X_test_multi)

model = MultinomialNB()
model.fit(X_train_vec, y_train_multi)
y_pred = model.predict(X_test_vec)

print("\n--- MultinomialNB + Count BoW (Multi-class) ---")
print(f"Accuracy: {accuracy_score(y_test_multi, y_pred):.2%}")
print(classification_report(y_test_multi, y_pred))
print(pd.DataFrame(confusion_matrix(y_test_multi, y_pred),
                   index=model.classes_, columns=model.classes_))


--- MultinomialNB + Count BoW (Multi-class) ---
Accuracy: 43.74%
              precision    recall  f1-score   support

      center       0.41      0.41      0.41      1941
        left       0.43      0.49      0.46      2098
       right       0.48      0.42      0.44      2088

    accuracy                           0.44      6127
   macro avg       0.44      0.44      0.44      6127
weighted avg       0.44      0.44      0.44      6127

        center  left  right
center     794   678    469
left       603  1018    477
right      562   658    868


2b Multinomial Naive Bayes + Count BoW (Binary)

In [11]:
vectorizer = CountVectorizer(max_features=1000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train_binary)
X_test_vec = vectorizer.transform(X_test_binary)

model = MultinomialNB()
model.fit(X_train_vec, y_train_binary)
y_pred = model.predict(X_test_vec)

print("\n--- MultinomialNB + Count BoW (Binary) ---")
print(f"Accuracy: {accuracy_score(y_test_binary, y_pred):.2%}")
print(classification_report(y_test_binary, y_pred))
print(pd.DataFrame(confusion_matrix(y_test_binary, y_pred),
                   index=model.classes_, columns=model.classes_))


--- MultinomialNB + Count BoW (Binary) ---
Accuracy: 61.71%
              precision    recall  f1-score   support

        left       0.61      0.67      0.64      2106
       right       0.63      0.56      0.60      2096

    accuracy                           0.62      4202
   macro avg       0.62      0.62      0.62      4202
weighted avg       0.62      0.62      0.62      4202

       left  right
left   1409    697
right   912   1184


3a Multinomial Naive Bayes + TF-IDF (Multi-class)

In [12]:
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train_multi)
X_test_vec = vectorizer.transform(X_test_multi)

model = MultinomialNB()
model.fit(X_train_vec, y_train_multi)
y_pred = model.predict(X_test_vec)

print("\n--- MultinomialNB + TF-IDF (Multi-class) ---")
print(f"Accuracy: {accuracy_score(y_test_multi, y_pred):.2%}")
print(classification_report(y_test_multi, y_pred))
print(pd.DataFrame(confusion_matrix(y_test_multi, y_pred),
                   index=model.classes_, columns=model.classes_))


--- MultinomialNB + TF-IDF (Multi-class) ---
Accuracy: 43.06%
              precision    recall  f1-score   support

      center       0.42      0.22      0.29      1941
        left       0.42      0.57      0.48      2098
       right       0.45      0.49      0.47      2088

    accuracy                           0.43      6127
   macro avg       0.43      0.43      0.41      6127
weighted avg       0.43      0.43      0.42      6127

        center  left  right
center     436   856    649
left       318  1187    593
right      292   781   1015


3b Multinomial Naive Bayes + TF-IDF (Binary)

In [13]:
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train_binary)
X_test_vec = vectorizer.transform(X_test_binary)

model = MultinomialNB()
model.fit(X_train_vec, y_train_binary)
y_pred = model.predict(X_test_vec)

print("\n--- MultinomialNB + TF-IDF (Binary) ---")
print(f"Accuracy: {accuracy_score(y_test_binary, y_pred):.2%}")
print(classification_report(y_test_binary, y_pred))
print(pd.DataFrame(confusion_matrix(y_test_binary, y_pred),
                   index=model.classes_, columns=model.classes_))


--- MultinomialNB + TF-IDF (Binary) ---
Accuracy: 60.99%
              precision    recall  f1-score   support

        left       0.60      0.66      0.63      2106
       right       0.62      0.56      0.59      2096

    accuracy                           0.61      4202
   macro avg       0.61      0.61      0.61      4202
weighted avg       0.61      0.61      0.61      4202

       left  right
left   1397    709
right   930   1166
